# Case 2 · Is the sales bootcamp worth it? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [1]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case2_controls_training/data/"
for candidate in ["data/", "cases/case2_controls_training/data/", "../../../cases/case2_controls_training/data/", "../../../../cases/case2_controls_training/data/"]:
    if os.path.exists(candidate + "reps_2025.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case2_controls_training/data/


## Stage 5 · Something you didn't know

> **From:** Jenna Park, Sales Operations
> **Re:** before you finalize the bootcamp analysis
>
> A few things you should know. Sign-up was voluntary, and the reps who signed up were mostly our **stars and reps with the biggest territories**; they wanted a shot at promotion. Our 2024 planning file has each rep's **2024 quota attainment** and **territory potential**, set before anyone knew about the bootcamp: `planning_2024.csv`.
>
> Also, on timing: **client meetings** and **manager ratings** are 2025 numbers, recorded *after* the bootcamp. Managers were told to give credit for "initiative", and attending the bootcamp counted. And the bootcamp's core module was on booking more first meetings with clients: attendees did log more meetings afterwards.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [2]:
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "reps_2025.csv")
print(df.shape)
df.head()

(1200, 9)


,rep_id,region,segment,tenure_years,attended_bootcamp_2025,client_meetings_2025,crm_logins_2025,manager_rating_2025,sales_2025
0,R0001,Central,SMB,2.8,0,122,176,1.5,723200.0
1,R0002,Central,Mid-market,3.7,1,199,187,3.3,1253700.0
2,R0003,West,SMB,1.6,1,103,210,2.2,374600.0
3,R0004,West,Mid-market,4.3,0,138,175,1.5,813100.0
4,R0005,Central,Mid-market,1.8,0,173,163,3.0,937800.0


In [3]:
plan = pd.read_csv(DATA + "planning_2024.csv")
d = df.merge(plan, on="rep_id")
d.head()

,rep_id,region,segment,tenure_years,attended_bootcamp_2025,client_meetings_2025,crm_logins_2025,manager_rating_2025,sales_2025,attainment_2024_pct,territory_potential_2024
0,R0001,Central,SMB,2.8,0,122,176,1.5,723200.0,90.0,0.76
1,R0002,Central,Mid-market,3.7,1,199,187,3.3,1253700.0,119.0,1.37
2,R0003,West,SMB,1.6,1,103,210,2.2,374600.0,106.0,0.40
3,R0004,West,Mid-market,4.3,0,138,175,1.5,813100.0,90.0,0.79
4,R0005,Central,Mid-market,1.8,0,173,163,3.0,937800.0,95.0,0.86


## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · How the estimate moves with the controls

Regress **log sales** on `attended_bootcamp_2025`, adding controls in the order below. Report the effect as a percentage: `exp(coefficient) − 1`. For each row, also note whether the variables you added were measured **before** or **after** the bootcamp.

| Controls | Bootcamp effect on sales | Added variables measured before or after? |
|---|---|---|
| 1. None | | — |
| 2. + tenure, segment, region | | |
| 3. Row 2 + 2024 attainment, territory potential | | |
| 4. Row 3 + client meetings | | |
| 5. Row 3 + manager rating | | |

Which row should you trust, and why do rows 4 and 5 move the estimate?

In [ ]:
import numpy as np
d["ls"] = np.log(d.sales_2025)
specs = {
    "1. none": "ls ~ attended_bootcamp_2025",
    "2. + tenure, segment, region": "ls ~ attended_bootcamp_2025 + tenure_years + C(segment) + C(region)",
    "3. + 2024 attainment, territory": "...",   # add np.log(attainment_2024_pct) + np.log(territory_potential_2024)
    "4. row 3 + client meetings": "...",
    "5. row 3 + manager rating": "...",
}
for name, f in specs.items():
    if f == "...":
        continue
    m = smf.ols(f, d).fit()
    print(f"{name:35s} {np.exp(m.params['attended_bootcamp_2025']) - 1:+.1%}")


### Step 2 · Look back: what did the 2025 bootcamp do? (ETT from 2025 data)

This step uses only the **2025 data** to estimate what the bootcamp did for the reps who attended it in February–March 2025: the effect on the treated (ETT). Nothing about 2026 yet.

Use **row 3's specification** from Step 1 (only variables measured before the bootcamp). Then run the **same regression** separately for juniors and for seniors. Juniors here means tenure < 3 **when they attended**, which for 2025 attendees is `tenure_years < 3`. Report each effect as `exp(coefficient) − 1`.

| Reps in the regression | N reps (of whom attended) | Bootcamp effect on sales |
|---|---|---|
| All reps | | |
| Juniors (tenure_years < 3) | | |
| Seniors (tenure_years ≥ 3) | | |

Which group does the bootcamp help?

In [ ]:
f3 = "..."   # paste your row 3 formula from Step 1
groups = {"all": d, "juniors": d[d.tenure_years < 3], "seniors": d[d.tenure_years >= 3]}
for name, sub in groups.items():
    # Your code here: fit smf.ols(f3, sub), then print len(sub), sub.attended_bootcamp_2025.sum()
    # and np.exp(coefficient on attended_bootcamp_2025) - 1
    pass


### Step 3 · Look ahead: predict 2026 profit

Now **predict** what next year's bootcamp would earn. Take the effects you **estimated** in Step 2 and apply them to the reps who have **not** attended yet. Each rep you send costs **\$15,000**; margin is **30%**. This is a forecast: your deployment will show the actual 2026 result.

The effect depends on a rep's tenure **when they are trained**. Next year's bootcamp is in January 2026, when every rep has one more year of tenure (`tenure_years + 1`). So a rep who is junior in 2026 has `tenure_years + 1 < 3`, i.e. `tenure_years < 2`.

| Untrained reps, by tenure in January 2026 | N | Average 2025 sales | Effect (from Step 2) | Extra margin per rep = 0.30 × effect × sales | Worth \$15,000? |
|---|---|---|---|---|---|
| Juniors (tenure_years + 1 < 3) | | | | | |
| Seniors (tenure_years + 1 ≥ 3) | | | | | |

Which group should go? Choose T, the January-2026 tenure cutoff, so that you send exactly that group. **Predicted 2026 profit** = Σ (extra margin − \$15,000) over the reps you send.

In [ ]:
# Your code here


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
MAX_TENURE = None          # send untrained reps with January-2026 tenure <= this; 0 = nobody  (e.g. 3)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("MAX_TENURE", MAX_TENURE), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              2")
print("round:             2  (Part II)")
print(f"decision:          {MAX_TENURE}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words).
- **Case:** `2`.  **Round:** `2` (Part II).
- Check the printed values before you submit. If your team submits more than once for the same round, only the **last** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.